# Deploy Sentiment Analysis Model sebagai REST API (FastAPI + ngrok)

Notebook ini memuat model hasil training dari `sentiment_analysis_lstm_training.ipynb`, membungkusnya jadi REST API dengan **FastAPI**, lalu meng-expose-nya ke internet lewat **ngrok** supaya bisa diakses dari luar Colab.

> **Prasyarat:** sudah menjalankan notebook training dan punya file `sentiment-model.pt` + `vocab.txt` tersimpan di Google Drive.


## 1. Install Dependencies

In [ ]:
!pip install fastapi uvicorn pyngrok pydantic torch torchtext spacy

In [ ]:
!python -m spacy download en_core_web_sm

## 2. Mount Google Drive

Untuk mengakses model dan vocabulary yang sudah disimpan sebelumnya.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Definisi Model & FastAPI App

Class model harus identik dengan yang dipakai saat training (`SentimentLSTM`), supaya `state_dict` yang dimuat cocok.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
from torchtext import data
import spacy
from pyngrok import ngrok
import uvicorn
import threading
import os

app = FastAPI(title='Sentiment Analysis API', version='1.0')
nlp = spacy.load('en_core_web_sm')

class SentimentLSTM(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = torch.nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                                  bidirectional=bidirectional, dropout=dropout)
        self.fc = torch.nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_embedded = torch.nn.utils.rnn.pack_padded_sequence(embedded, text_lengths.to('cpu'))
        packed_output, (hidden, _) = self.rnn(packed_embedded)
        hidden = self.dropout(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        return self.fc(hidden)

## 4. Load Vocabulary & Model dari Drive

In [ ]:
MODEL_DIR = '/content/drive/MyDrive/AI-Engineer/NLP/models/'

def load_vocab(vocab_file_path):
    vocab = {}
    with open(vocab_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line:
                continue
            word, index = line.split('\t')
            vocab[word] = int(index)
    return vocab

def load_model():
    INPUT_DIM = 25002       # ukuran vocab saat training (25,000 + <unk> + <pad>)
    EMBEDDING_DIM = 100
    HIDDEN_DIM = 256
    OUTPUT_DIM = 1
    N_LAYERS = 2
    BIDIRECTIONAL = True
    DROPOUT = 0.5
    PAD_IDX = 1             # index token <pad> saat training

    model_path = os.path.join(MODEL_DIR, 'sentiment-model.pt')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model tidak ditemukan di {model_path}')

    model = SentimentLSTM(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM,
                           N_LAYERS, BIDIRECTIONAL, DROPOUT, PAD_IDX)
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
    model.eval()
    return model

vocab = load_vocab(os.path.join(MODEL_DIR, 'vocab.txt'))
model = load_model()

def predict_sentiment(sentence: str):
    tokenized = [tok.text for tok in nlp.tokenizer(sentence)]
    indexed = [vocab.get(t, vocab.get('<unk>', 0)) for t in tokenized]
    length = [len(indexed)]
    tensor = torch.LongTensor(indexed).unsqueeze(1)
    length_tensor = torch.LongTensor(length)
    with torch.no_grad():
        prediction = torch.sigmoid(model(tensor, length_tensor))
    return prediction.item()

## 5. API Routes

- `GET /` — health check
- `POST /predict/` — menerima kalimat, mengembalikan skor sentimen dan label kategori (`very positive` / `positive` / `neutral` / `negative`)

In [ ]:
class SentimentRequest(BaseModel):
    sentence: str

@app.get('/')
def read_root():
    return {'message': 'Welcome to the Sentiment Analysis API'}

@app.post('/predict/')
def analyze_sentiment(request: SentimentRequest):
    try:
        score = predict_sentiment(request.sentence)
        if score >= 0.80:
            sentiment = 'very positive'
        elif score >= 0.5:
            sentiment = 'positive'
        elif score >= 0.25:
            sentiment = 'neutral'
        else:
            sentiment = 'negative'
        return {'sentence': request.sentence, 'sentiment': sentiment, 'score': score}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Error occurred during prediction: {e}')

## 6. Konfigurasi ngrok

> ⚠️ **Keamanan:** JANGAN pernah hardcode authtoken ngrok langsung di notebook, terutama kalau notebook ini akan di-share atau di-push ke repo publik — token yang bocor bisa dipakai orang lain memakai kuota akun ngrok-mu. Simpan token sebagai Colab Secret (ikon 🔑 di sidebar kiri Colab) dengan nama `NGROK_AUTHTOKEN`, lalu ambil lewat `google.colab.userdata`.

In [ ]:
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')  # diset lewat Colab Secrets, bukan hardcode
!ngrok config add-authtoken $NGROK_AUTHTOKEN

## 7. Jalankan Server & Buka Tunnel

In [ ]:
def run_app():
    uvicorn.run(app, host='0.0.0.0', port=8000)

ngrok_tunnel = ngrok.connect(8000)
print(f'Public URL: {ngrok_tunnel.public_url}')

thread = threading.Thread(target=run_app)
thread.start()

Public URL: https://<random-subdomain>.ngrok-free.dev


## 8. Tes API

Contoh hasil pemanggilan endpoint `/predict/` (URL diganti placeholder — URL asli ngrok berubah setiap kali tunnel baru dibuka):

In [ ]:
import requests

url = f'{ngrok_tunnel.public_url}/predict/'
data = {'sentence': 'This film is great'}
response = requests.post(url, json=data)
print(response.json())

{'sentence': 'This film is great', 'sentiment': 'very positive', 'score': 0.9846353530883789}


---

*Notebook ini disusun sebagai bagian dari sesi rubythalib.ai AI Engineer Bootcamp, dengan bimbingan mentor Muhammad Ikhwan Fathulloh.*